In [0]:
import pandas as pd
import numpy as np

amazon = spark.table("workspace.amazon_data.amazon").toPandas()

amazon["OrderDate"] = pd.to_datetime(amazon["OrderDate"])

customer_360 = spark.table(
    "workspace.default.amazon_customer_360"
).toPandas()

### Calculate Recency

In [0]:
analysis_date = amazon["OrderDate"].max() + pd.Timedelta(days=1)

rfm = (
    amazon.groupby("CustomerID")
    .agg(
        Recency=("OrderDate", lambda x:
                 (analysis_date - x.max()).days),
        Frequency=("OrderID", "nunique"),
        Monetary=("TotalAmount", "sum")
    )
    .reset_index()
)

rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,CUST000001,157,1,446.48
1,CUST000002,1063,3,1389.93
2,CUST000003,1463,1,1116.91
3,CUST000004,196,3,2660.06
4,CUST000005,640,1,1089.56


### create RFM score

In [0]:
rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

In [0]:
rfm["RFM_Score"] = (
    rfm["R_Score"]
    + rfm["F_Score"]
    + rfm["M_Score"]
)

### create customer segments

In [0]:
def assign_segment(row):
    r = row["R_Score"]
    f = row["F_Score"]
    m = row["M_Score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"

    elif r >= 3 and f >= 4:
        return "Loyal Customers"

    elif r >= 4 and f >= 2:
        return "Potential Loyalists"

    elif r <= 2 and f >= 3:
        return "At Risk"

    elif r <= 2 and f <= 2:
        return "Lost Customers"

    else:
        return "Regular Customers"


rfm["CustomerSegment"] = rfm.apply(
    assign_segment,
    axis=1
)

### Analyze segments

In [0]:
rfm["CustomerSegment"].value_counts()

CustomerSegment
Lost Customers         10081
At Risk                 7212
Champions               7129
Regular Customers       6561
Loyal Customers         6480
Potential Loyalists     5770
Name: count, dtype: int64

In [0]:
rfm.groupby(
    "CustomerSegment"
)["Monetary"].agg(
    ["count", "sum", "mean"]
).sort_values(
    "sum",
    ascending=False
)

,count,sum,mean
CustomerSegment,,,
Champions,7129,29290057.89,4108.578747
At Risk,7212,17480947.85,2423.869641
Loyal Customers,6480,16310984.31,2517.127208
Lost Customers,10081,10761355.27,1067.488867
Potential Loyalists,5770,9627003.07,1668.458071
Regular Customers,6561,8355299.53,1273.479581


### Save RFM table

In [0]:
spark.createDataFrame(
    rfm
).write.mode(
    "overwrite"
).saveAsTable(
    "amazon_rfm"
)